In [15]:
import pandas as pd

df = pd.read_csv("blogs.csv", low_memory=False)
df = df[['Data', 'Labels']]
df = df.dropna()

### Text Cleaning

In [16]:
import re

df['Data'] = df['Data'].astype(str)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df['Data'] = df['Data'].apply(clean_text)

### Remove Empty row

In [11]:
df = df[df['Data'].str.strip() != ""]

### Remove Rare Labels

In [17]:
label_counts = df['Labels'].value_counts()
valid_labels = label_counts[label_counts >= 50].index
df = df[df['Labels'].isin(valid_labels)]

### TF-IDF

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)

X = tfidf.fit_transform(df['Data'])  # IMPORTANT
y = df['Labels']

### Train-Test Split

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Naive Bayes Model

In [20]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


### Prediction

In [21]:
y_pred = model.predict(X_test)

### Evaluation

In [22]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7543859649122807
                    precision    recall  f1-score   support

                         0.59      0.80      0.68        30
              Inc.       1.00      0.88      0.94        17
               and       1.00      0.25      0.40        16
               but       0.00      0.00      0.00         8
       alt.atheism       0.72      1.00      0.84        29
talk.politics.misc       1.00      1.00      1.00        14

          accuracy                           0.75       114
         macro avg       0.72      0.66      0.64       114
      weighted avg       0.75      0.75      0.71       114



C:\Users\Akhlaque Alam\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Akhlaque Alam\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Akhlaque Alam\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this b

### Sentiment Analysis

In [27]:
from textblob import TextBlob

def get_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    
    if polarity > 0:
        return "Positive"
    elif polarity < 0:
        return "Negative"
    else:
        return "Neutral"

df['Sentiment'] = df['Data'].apply(get_sentiment)

In [28]:
df[['Data', 'Sentiment']].head()

,Data,Sentiment
0,path cantaloupesrvcscmuedumagnesiumclubcccmued...,Positive
1,newsgroups altatheism path cantaloupesrvcscmue...,Negative
2,path cantaloupesrvcscmuedudasnewsharvardedunoc...,Positive
3,path cantaloupesrvcscmuedumagnesiumclubcccmued...,Positive
4,xref cantaloupesrvcscmuedu altatheism talkreli...,Positive


In [29]:
df['Sentiment'].value_counts()

Sentiment
Neutral     327
Positive    158
Negative     84
Name: count, dtype: int64

### Category vs Sentiment

In [30]:
import pandas as pd

pd.crosstab(df['Labels'], df['Sentiment'])

Sentiment,Negative,Neutral,Positive
Labels,,,
,24,97,44
Inc.,0,77,2
and,16,49,24
but,13,26,19
alt.atheism,31,17,69
talk.politics.misc,0,61,0
